In [12]:
!pip install cryptography pyotp

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: C:\Python313\python.exe -m pip install --upgrade pip


In [ ]:
import os
import base64
import hashlib
import pyotp
from cryptography.fernet import Fernet
from cryptography.hazmat.primitives.asymmetric import rsa, padding
from cryptography.hazmat.primitives import serialization, hashes
from cryptography.hazmat.backends import default_backend

# -------------------------------
# USER DATABASE (DEMO PURPOSE)
# -------------------------------
users = {
    "admin": {
        "password": hashlib.sha256("admin123".encode()).hexdigest(),
        "role": "admin",
        "2fa_secret": pyotp.random_base32()
    },
    "user": {
        "password": hashlib.sha256("user123".encode()).hexdigest(),
        "role": "user",
        "2fa_secret": pyotp.random_base32()
    }
}

# -------------------------------
# AUTHENTICATION SYSTEM
# -------------------------------
def login():
    username = input("Enter username: ")
    password = input("Enter password: ")

    if username in users:
        hashed = hashlib.sha256(password.encode()).hexdigest()
        if hashed == users[username]["password"]:
            print("Password correct!")

            # 2FA
            totp = pyotp.TOTP(users[username]["2fa_secret"])
            print("Your 2FA code (for demo):", totp.now())

            code = input("Enter 2FA code: ")
            if totp.verify(code):
                print("Login successful!")
                return username
            else:
                print("Invalid 2FA!")
        else:
            print("Wrong password!")
    else:
        print("User not found!")

    return None

# -------------------------------
# AES ENCRYPTION (FERNET)
# -------------------------------
def generate_aes_key():
    key = Fernet.generate_key()
    with open("aes.key", "wb") as f:
        f.write(key)
    print("AES key generated!")
    return key

def load_aes_key():
    return open("aes.key", "rb").read()

def aes_encrypt(text):
    key = load_aes_key()
    f = Fernet(key)
    encrypted = f.encrypt(text.encode())
    print("Encrypted:", encrypted)
    return encrypted

def aes_decrypt(token):
    key = load_aes_key()
    f = Fernet(key)
    decrypted = f.decrypt(token).decode()
    print("Decrypted:", decrypted)
    return decrypted

# -------------------------------
# RSA KEY GENERATION
# -------------------------------
def generate_rsa_keys():
    private_key = rsa.generate_private_key(
        public_exponent=65537,
        key_size=2048
    )

    public_key = private_key.public_key()

    # Save keys
    with open("private.pem", "wb") as f:
        f.write(private_key.private_bytes(
            encoding=serialization.Encoding.PEM,
            format=serialization.PrivateFormat.PKCS8,
            encryption_algorithm=serialization.NoEncryption()
        ))

    with open("public.pem", "wb") as f:
        f.write(public_key.public_bytes(
            encoding=serialization.Encoding.PEM,
            format=serialization.PublicFormat.SubjectPublicKeyInfo
        ))

    print("RSA keys generated!")

# -------------------------------
# RSA ENCRYPT / DECRYPT
# -------------------------------
def rsa_encrypt(message):
    with open("public.pem", "rb") as f:
        public_key = serialization.load_pem_public_key(f.read())

    encrypted = public_key.encrypt(
        message.encode(),
        padding.OAEP(
            mgf=padding.MGF1(algorithm=hashes.SHA256()),
            algorithm=hashes.SHA256(),
            label=None
        )
    )

    encoded = base64.b64encode(encrypted).decode()
    print("Encrypted:", encoded)
    return encoded

def rsa_decrypt(ciphertext):
    with open("private.pem", "rb") as f:
        private_key = serialization.load_pem_private_key(f.read(), password=None)

    decrypted = private_key.decrypt(
        ciphertext,
        padding.OAEP(
            mgf=padding.MGF1(algorithm=hashes.SHA256()),
            algorithm=hashes.SHA256(),
            label=None
        )
    )

    print("Decrypted:", decrypted.decode())
    return decrypted.decode()

# -------------------------------
# HASHING FUNCTIONS
# -------------------------------
def hash_data(data):
    print("MD5:", hashlib.md5(data.encode()).hexdigest())
    print("SHA-256:", hashlib.sha256(data.encode()).hexdigest())
    print("SHA-512:", hashlib.sha512(data.encode()).hexdigest())

# -------------------------------
# FILE INTEGRITY CHECK
# -------------------------------
def file_hash(filepath):
    with open(filepath, "rb") as f:
        content = f.read()
        return hashlib.sha256(content).hexdigest()

def check_integrity(file1, file2):
    if file_hash(file1) == file_hash(file2):
        print("Files are identical (Integrity OK)")
    else:
        print("Files are different (Tampered!)")

# -------------------------------
# MAIN MENU
# -------------------------------
def main():
    user = login()
    if not user:
        return

    while True:
        print("\n--- MENU ---")
        print("1. Generate AES Key")
        print("2. AES Encrypt Text")
        print("3. AES Decrypt Text")
        print("4. Generate RSA Keys")
        print("5. RSA Encrypt")
        print("6. RSA Decrypt")
        print("7. Hash Data")
        print("8. File Integrity Check")
        print("9. Exit")

        choice = input("Enter choice: ")

        if choice == "1":
            generate_aes_key()

        elif choice == "2":
            text = input("Enter text: ")
            aes_encrypt(text)

        elif choice == "3":
            token = input("Paste encrypted text: ").encode('utf-8')
            aes_decrypt(token)

        elif choice == "4":
            generate_rsa_keys()

        elif choice == "5":
            msg = input("Enter message: ")
            rsa_encrypt(msg)

        elif choice == "6":
            ct = input("Paste encrypted text: ")
            ciphertext = base64.b64decode(ct)
            rsa_decrypt(ciphertext)

        elif choice == "7":
            data = input("Enter data: ")
            hash_data(data)

        elif choice == "8":
            f1 = input("File 1 path: ")
            f2 = input("File 2 path: ")
            check_integrity(f1, f2)

        elif choice == "9":
            break

        else:
            print("Invalid choice!")

# RUN
main()

Password correct!
Your 2FA code (for demo): 716090
Invalid 2FA!
